# 01 Explore: Karnataka daily wind and solar, gaps, flags, NASA weather join

Exploration only: no features and no models. Everything is read through the `solarwind` package from `data/processed` and `data/raw` (run `dvc pull` first on a fresh clone).

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from solarwind.config import get_settings, load_sites
from solarwind.data.cea.build_daily import missing_days
from solarwind.data.join import join_with_weather, source_weather
from solarwind.data.tables import load_daily, load_flags, load_parse_failures

# settings paths are relative to the repository root
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(root)

settings = get_settings()
sites = load_sites(settings.sites_csv)
daily = load_daily(settings)
print("repo root:", root)
print(
    "daily rows:",
    len(daily),
    "| first",
    daily["date"].min().date(),
    "| last",
    daily["date"].max().date(),
)
pd.DataFrame([s.model_dump() for s in sites])

## 1. Karnataka daily wind and solar over time

`wind_mu` and `solar_mu` are the CEA `Generation` sheet values (state control plus ISGS), in million units (MU), net. Lines break where days are missing.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
full_index = pd.date_range(daily["date"].min(), daily["date"].max())
series = daily.set_index("date").reindex(full_index)
axes[0].plot(series.index, series["wind_mu"], color="tab:blue", linewidth=0.9)
axes[0].set_ylabel("wind (MU/day)")
axes[1].plot(series.index, series["solar_mu"], color="tab:orange", linewidth=0.9)
axes[1].set_ylabel("solar (MU/day)")
axes[0].set_title("Karnataka daily generation (CEA Generation sheet)")
fig.tight_layout()
daily[["wind_mu", "solar_mu"]].describe()

## 2. Missing days by month, and the rejected files

A missing day is a date between the first and last report with no row in the table. That includes dates the CEA portal does not list and the files the parser rejected.

In [ ]:
gaps = missing_days(daily)
by_month = gaps.to_series().groupby(gaps.to_period("M")).size()
print("missing days:", len(gaps))
ax = by_month.plot(kind="bar", figsize=(10, 3.5), color="tab:red")
ax.set_ylabel("missing days")
ax.set_title("Missing days by month")
by_month.to_frame("missing_days")

In [ ]:
failures = load_parse_failures(settings)
print("rejected files:", len(failures))
failures

## 3. Flagged dates

Flags are logged by the build, never fixed. Each block shows the flagged date with the two days either side (a date missing from the table has no row).

In [ ]:
flags = load_flags(settings)
print("flags:", len(flags))
flags

In [ ]:
window = pd.Timedelta(days=2)
for flagged in flags["date"].drop_duplicates().sort_values():
    near = daily[(daily["date"] >= flagged - window) & (daily["date"] <= flagged + window)]
    print(f"--- around {flagged.date()}: " + "; ".join(flags.loc[flags["date"] == flagged, "flag"]))
    print(near.to_string(index=False))

## 4. Join with NASA POWER weather

The solar target is joined to the mean weather of the four solar sites and the wind target to the mean of the four wind sites, with the equal weights from `config/sites.csv`. A joined day needs a CEA row and weather for that date.

In [ ]:
solar_weather = source_weather(settings, sites, "solar")
wind_weather = source_weather(settings, sites, "wind")
joined = join_with_weather(daily, solar_weather, wind_weather)
cea_only = daily.loc[~daily["date"].isin(joined["date"]), "date"]
print("CEA range    :", daily["date"].min().date(), "to", daily["date"].max().date())
print(
    "NASA range   :", solar_weather["date"].min().date(), "to", solar_weather["date"].max().date()
)
print("rows         : CEA", len(daily), "| NASA", len(solar_weather), "| joined", len(joined))
print("CEA dates without weather:", len(cea_only), "->", cea_only.dt.date.astype(str).tolist())
print("weather NaN cells:", int(joined.filter(regex="^(solar|wind)_[A-Z]").isna().sum().sum()))
joined.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
pairs = [
    (
        axes[0],
        "solar_ALLSKY_SFC_SW_DWN",
        "solar_mu",
        "tab:orange",
        "irradiance (kWh/m2/day)",
        "solar (MU/day)",
    ),
    (axes[1], "wind_WS50M", "wind_mu", "tab:blue", "wind speed at 50 m (m/s)", "wind (MU/day)"),
]
for ax, x, y, color, xlabel, ylabel in pairs:
    ax.scatter(joined[x], joined[y], s=8, alpha=0.5, color=color)
    r = joined[x].corr(joined[y])
    ax.set(xlabel=xlabel, ylabel=ylabel, title=f"{y} vs {x} (r = {r:.2f})")
fig.tight_layout()